In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    "Данные_для_курсовои_Классическое_МО.xlsx"
)
df = df.drop(columns=["Unnamed: 0"])
df = df.fillna(df.median())

In [2]:
targets = ["IC50, mM", "CC50, mM", "SI"]
X = df.drop(columns=targets)

y = (df["IC50, mM"] > df["IC50, mM"].median()).astype(int)

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42, stratify=y) #гиперпараметр stratify используем чтобы доля классов сохранялась

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score)


lr = LogisticRegression(max_iter=5000, C=1.0) #выбрано оптимальное число итераций обучения и оптимальный штраф за слишком сложную модель

lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print("Logistic Regression")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))

Logistic Regression
Accuracy: 0.4975124378109453
F1: 0.6644518272425249
ROC-AUC: 0.5


Модель логистической регресии пока демонстрирует не самые лучшие значения, модель все еще слаба

Проверим модель случайного леса

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5) #глубина оптимальная, нету недообучения и риска переобучения
                                                                                #min_samples_split делает деревья менее сложными борется с переобучением
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Random Forest")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))


Random Forest
Accuracy: 0.7014925373134329
F1: 0.7222222222222222
ROC-AUC: 0.701881188118812


Данные параметры уже демонстрируют что такая модель очень хорошо понимает наши признаки и действительно находит закономерности, тем более с высокой точностью

Посмотрим на градиент бустинг

In [6]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42, max_depth=10, min_samples_split=5)   #глубина оптимальная, нету недообучения и риска переобучения
                                                                                     #min_samples_split делает деревья менее сложными борется с переобучением
gb.fit(X_train, y_train)

pred = gb.predict(X_test)

print("GradientBoostingClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))



GradientBoostingClassifier
Accuracy: 0.681592039800995
F1: 0.7064220183486238
ROC-AUC: 0.6820297029702971


Получились метрики чуть хуже, чем для случайного леса, но все еще отличные

Пока наша лучшая модель - модель случайного леса. Посмотрим как она работает на 9 наших избранных признаках

In [7]:

selected_features = [
    "VSA_EState8",
    "VSA_EState6",
    "VSA_EState4",
    "MolLogP",
    "qed",
    "FractionCSP3",
    "NumSaturatedHeterocycles",
    "NumSaturatedCarbocycles",
    "NumAliphaticCarbocycles"
]

X_small = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=5)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

In [9]:
print("RandomForestClassifier")

print("Accuracy:", accuracy_score(y_test, pred))

print("F1:", f1_score(y_test, pred))

print("ROC-AUC:", roc_auc_score(y_test, pred))


RandomForestClassifier
Accuracy: 0.7114427860696517
F1: 0.71
ROC-AUC: 0.7124801744647106


Мы сократили число признаков с 211 всего до 9, при этом качество ROC-AUC и Accuracy мало того что не упало но даже немного выросло, что подтверждает высокую информативность наших 9 признаков, выявленных ранее на этапе EDA. Вероятно причиной является то, что алгоритм распознал очень сильную взаимосвязь этих 9 признаков с IC50 когда исключили прочие дескрипторы создающие лишний шум, что сделало модель более устойчивой